# Experiment 6: lower learning rate

Second attempt. The first run aborted on its own determinism check: the same
configuration that scored 0.962141 in experiment 4 scored 0.962067 here, a delta of
7.4e-5 against an assertion threshold of 1e-6.

## What went wrong, and what changed

LightGBM is not reproducible by default. Its histogram construction sums
floating-point gradients across threads, and the order of that summation depends on
thread scheduling, so two runs of an identical configuration on the same machine can
differ in the last few digits. The divergence compounds with tree count, which is why
experiment 2's 100-tree check came within 3e-7 and passed while this 1000-tree check
came within 7.4e-5 and failed. **The 100-tree check was never exact either.** It was
read as exact and reported as such, and that was wrong.

Running the adversarial validation notebook concurrently made it worse by putting two
LightGBM jobs in contention for the same cores.

Two fixes:

1. `deterministic=True`, `force_row_wise=True`, and a fixed `n_jobs`, which together
   pin the histogram construction order. This costs some speed and buys exact
   reproducibility.
2. The determinism check now runs **inside this notebook**, training the same
   configuration twice and requiring the two to be bit-identical. Comparing against a
   number produced by a different notebook under different settings was never a clean
   test.

Nothing else runs while this does.

## Reading the older ledger rows

Experiments 1 to 5 were produced before these flags, so their CV values carry roughly
1e-4 of run-to-run noise. That does not disturb any conclusion drawn from them, since
the tree-count gain was +0.0072, about a hundred times larger. It does mean
distinctions at the fourth decimal between those rows are not meaningful, including
the 0.000309 that separated 1000 trees from 2000.

## The variable under test

Learning rate. Tree count follows by `n_estimators = round(100 / lr)` so the product
stays fixed, which keeps this one decision rather than two free choices. Everything
else stays at anchor settings, seed 42, identical folds.

In [1]:
import csv
import time
from datetime import datetime, timezone
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_SPLITS = 5
TARGET = "addicted_label"
ID = "id"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]

# Pins the floating-point reduction order. Without these LightGBM is not reproducible.
LGB_KW = dict(random_state=SEED, verbose=-1, deterministic=True,
              force_row_wise=True, n_jobs=4)

LR_GRID = [0.1, 0.05, 0.03]
BUDGET = 100

print("lightgbm", lgb.__version__)
for lr in LR_GRID:
    print(f"  lr={lr}  ->  n_estimators={round(BUDGET / lr)}")

lightgbm 4.7.0
  lr=0.1  ->  n_estimators=1000
  lr=0.05  ->  n_estimators=2000
  lr=0.03  ->  n_estimators=3333


In [2]:
def locate():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "raw" / "train.csv").exists():
            return base, base / "data" / "raw"
    kag = Path("/kaggle/input/playground-series-s6e8")
    if (kag / "train.csv").exists():
        return Path("/kaggle/working"), kag
    raise FileNotFoundError("could not find train.csv")


REPO, RAW = locate()
SUB_DIR = REPO / "submissions"
OOF_DIR = REPO / "artifacts" / "oof"
SUB_DIR.mkdir(parents=True, exist_ok=True)
OOF_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(RAW / "train.csv")
test = pd.read_csv(RAW / "test.csv")
sample = pd.read_csv(RAW / "sample_submission.csv")

FEATURES = [c for c in train.columns if c not in (ID, TARGET)]
for c in CAT_COLS:
    levels = pd.Categorical(pd.concat([train[c], test[c]], ignore_index=True)).categories
    train[c] = pd.Categorical(train[c], categories=levels)
    test[c] = pd.Categorical(test[c], categories=levels)

y = train[TARGET].to_numpy()

assert ID not in FEATURES, "id must never be a feature"
assert TARGET not in FEATURES, "target must never be a feature"
assert not (set(train[ID]) & set(test[ID])), "train and test ids overlap"
assert list(FEATURES) == [c for c in test.columns if c != ID], "train/test feature mismatch"

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = np.full(len(train), -1, dtype=int)
for i, (_, va) in enumerate(skf.split(train, y)):
    folds[va] = i
assert (folds >= 0).all()

print(f"{len(train):,} train rows, {len(FEATURES)} features, {N_SPLITS} folds")

691,369 train rows, 12 features, 5 folds


In [3]:
def run_level(lr, n_estimators):
    oof = np.zeros(len(train), dtype=float)
    test_pred = np.zeros(len(test), dtype=float)
    fold_scores = []
    t0 = time.time()

    for f in range(N_SPLITS):
        tr_m, va_m = folds != f, folds == f
        model = lgb.LGBMClassifier(n_estimators=n_estimators, learning_rate=lr, **LGB_KW)
        model.fit(train.loc[tr_m, FEATURES], y[tr_m])
        p_va = model.predict_proba(train.loc[va_m, FEATURES])[:, 1]
        oof[va_m] = p_va
        test_pred += model.predict_proba(test[FEATURES])[:, 1] / N_SPLITS
        fold_scores.append(roc_auc_score(y[va_m], p_va))

    return {"lr": lr, "n_estimators": n_estimators,
            "cv_mean": float(np.mean(fold_scores)), "cv_std": float(np.std(fold_scores)),
            "pooled": float(roc_auc_score(y, oof)), "secs": time.time() - t0,
            "oof": oof, "test_pred": test_pred}

## Determinism check, run first

The reference configuration is trained twice. The two runs must agree bit for bit. If
they do not, the flags are not doing their job and nothing downstream is comparable, so
this aborts before spending time on the sweep.

In [4]:
ref_a = run_level(0.1, 1000)
ref_b = run_level(0.1, 1000)
delta = abs(ref_a["cv_mean"] - ref_b["cv_mean"])
oof_max = float(np.abs(ref_a["oof"] - ref_b["oof"]).max())

print(f"run A CV : {ref_a['cv_mean']:.9f}")
print(f"run B CV : {ref_b['cv_mean']:.9f}")
print(f"delta    : {delta:.12f}")
print(f"largest per-row OOF difference: {oof_max:.3e}")

assert delta == 0.0, f"LightGBM still not deterministic: delta {delta:.3e}"
assert oof_max == 0.0, f"OOF predictions differ by up to {oof_max:.3e}"
print("\nbit-identical. Comparisons below are trustworthy.")

run A CV : 0.962198237
run B CV : 0.962198237
delta    : 0.000000000000
largest per-row OOF difference: 0.000e+00

bit-identical. Comparisons below are trustworthy.


## Sweep

In [5]:
results = [dict(ref_a)]
results[0]["secs"] = ref_a["secs"]
for lr in LR_GRID[1:]:
    r = run_level(lr, round(BUDGET / lr))
    results.append(r)

for r in results:
    print(f"  lr={r['lr']:<5} n={r['n_estimators']:>5}  cv={r['cv_mean']:.6f} "
          f"+/- {r['cv_std']:.6f}  pooled={r['pooled']:.6f}  {r['secs']:.0f}s")

  lr=0.1   n= 1000  cv=0.962198 +/- 0.000816  pooled=0.962195  173s
  lr=0.05  n= 2000  cv=0.963210 +/- 0.000591  pooled=0.963209  347s
  lr=0.03  n= 3333  cv=0.963275 +/- 0.000549  pooled=0.963274  563s


## Read it

The baseline for comparison is this notebook's own lr=0.1 run, not experiment 4's
number, because experiment 4 predates the determinism flags. The bar for calling a
change real is the fold spread, not zero.

In [6]:
base = results[0]["cv_mean"]
best = max(results, key=lambda r: r["cv_mean"])
print(f"{'lr':>6} {'n_est':>7} {'cv':>10} {'sd':>9} {'vs lr0.1':>10} {'vs sd':>7} {'secs':>6}")
print("-" * 60)
for r in results:
    gain = r["cv_mean"] - base
    in_sd = gain / r["cv_std"] if r["cv_std"] else float("nan")
    print(f"{r['lr']:>6} {r['n_estimators']:>7} {r['cv_mean']:>10.6f} {r['cv_std']:>9.6f} "
          f"{gain:>+10.6f} {in_sd:>7.1f} {r['secs']:>6.0f}")

gain = best["cv_mean"] - base
print(f"\nbest: lr={best['lr']} at {best['cv_mean']:.6f}, {gain:+.6f} vs lr 0.1")
if gain < best["cv_std"]:
    print("\nVERDICT: inside one fold standard deviation. Learning rate is not where "
          "the remaining gap lives.\nStop tuning. Move to model diversity and blending.")
else:
    print(f"\nVERDICT: {gain / best['cv_std']:.1f} fold sd. Real. Confirm across seeds "
          "before it goes in a final blend.")
if best["lr"] == LR_GRID[-1]:
    print("Best sits at the edge of the grid, so the curve has not turned over.")

    lr   n_est         cv        sd   vs lr0.1   vs sd   secs
------------------------------------------------------------
   0.1    1000   0.962198  0.000816  +0.000000     0.0    173
  0.05    2000   0.963210  0.000591  +0.001012     1.7    347
  0.03    3333   0.963275  0.000549  +0.001076     2.0    563

best: lr=0.03 at 0.963275, +0.001076 vs lr 0.1

VERDICT: 2.0 fold sd. Real. Confirm across seeds before it goes in a final blend.
Best sits at the edge of the grid, so the curve has not turned over.


In [7]:
LEDGER = REPO / "experiments.csv"
COLUMNS = ["id", "utc", "name", "cv_mean", "cv_std", "folds",
           "lb_public", "lb_private", "submitted", "notes"]

rows = []
if LEDGER.exists():
    with LEDGER.open(newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
next_id = max((int(r["id"]) for r in rows), default=0) + 1

for r in results:
    tag = f"lgbm_lr{str(r['lr']).replace('.', '')}_n{r['n_estimators']}_seed{SEED}"
    np.save(OOF_DIR / f"{tag}.npy", r["oof"])
    sub = sample.copy()
    sub[TARGET] = r["test_pred"]
    sub.to_csv(SUB_DIR / f"{tag}.csv", index=False)

    rows.append({
        "id": str(next_id), "utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
        "name": f"lgbm_lr{str(r['lr']).replace('.', '')}",
        "cv_mean": f"{r['cv_mean']:.6f}", "cv_std": f"{r['cv_std']:.6f}",
        "folds": str(N_SPLITS), "lb_public": "", "lb_private": "", "submitted": "no",
        "notes": (f"lr={r['lr']}, n_estimators={r['n_estimators']} holding lr*n=100, "
                  f"DETERMINISTIC flags on, not comparable at 4dp to rows 1-5"),
    })
    next_id += 1

with LEDGER.open("w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=COLUMNS)
    w.writeheader()
    w.writerows({c: r.get(c, "") for c in COLUMNS} for r in rows)

pd.read_csv(LEDGER)

,id,utc,name,cv_mean,cv_std,folds,lb_public,lb_private,submitted,notes
0,1,2026-08-04 02:15,lgbm_default_anchor,0.954947,0.000645,5,0.95594,NaN,yes,"untuned lgbm defaults, raw features, native ca..."
1,2,2026-08-04 05:29,lgbm_trees100,0.954947,0.000645,5,NaN,NaN,no,"reproducibility re-run of exp 1, same config, ..."
2,3,2026-08-04 05:29,lgbm_trees300,0.960605,0.000688,5,NaN,NaN,no,"n_estimators=300, lr default 0.1, no early sto..."
3,4,2026-08-04 05:29,lgbm_trees1000,0.962141,0.000859,5,0.96435,NaN,yes,"n_estimators=1000, lr default 0.1, no early st..."
4,5,2026-08-04 05:29,lgbm_trees2000,0.961832,0.000952,5,NaN,NaN,no,"n_estimators=2000, lr default 0.1, no early st..."
5,6,2026-08-04 06:34,lgbm_lr01,0.962198,0.000816,5,NaN,NaN,no,"lr=0.1, n_estimators=1000 holding lr*n=100, DE..."
6,7,2026-08-04 06:34,lgbm_lr005,0.963210,0.000591,5,NaN,NaN,no,"lr=0.05, n_estimators=2000 holding lr*n=100, D..."
7,8,2026-08-04 06:34,lgbm_lr003,0.963275,0.000549,5,NaN,NaN,no,"lr=0.03, n_estimators=3333 holding lr*n=100, D..."
